In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Section 1 Defining Parameters and Bounds.

Values were used from previous experiments, bounds are arbritrary almost in this model. But were based off of previous calibration of the model.

In [3]:
class TomgroParams:
    def __init__(self, **kwargs):
        # Calibrated parameters
        self.N_m = 0.5
        self.delta = 0.03
        self.Beta = 0.4
        self.N_b = 16
        self.LAI_max = 3.5
        self.alpha_f = 0.8
        self.vartheta = 0.135
        self.T_Crit = 24.4      # fruit abortion (g) -- literature value, leave alone
        self.V_max = 8
        self.N_FF = 22
        self.kappa_f = 5
        self.D_Fmax = 0.04

        # Node development (f_N)
        self.T_base_N = 9
        self.T_mid_N  = 12
        self.T_opt_N  = 25
        self.Tmax_N   = 35

        # Photosynthesis (Pgred)
        self.T_base_Pg = 9
        self.T_mid_Pg  = 12
        self.T_opt_Pg  = 25
        self.Tmax_Pg   = 35

        #Fruit growth
        self.T_base_fruit=9
        self.Tmid_fruit=12
        self.T_opt_fruit=24
        self.T_max_fruit=35

        # LAI temperature limitation (lambda_Td)
        self.T_Crit_LAI = 24.4
        self.Tmax_LAI   = 35

        # Fixed parameters
        self.E = 0.7
        self.r_m = 0.016
        self.Q10 = 1.4
        self.p1 = 2
        self.K = 0.58
        self.m = 0.1
        self.Q_e = 0.0645
        self.D = 2.593
        self.tau = 0.0693
        #Initial Conditions
        self.rho = 2.7529 # Plant density
        self.N0= 9  #initial Nodes
        self.LAI0= 0.2 #Initial LAI
        self.W0= 0.5  #Initial Crop Weight
        self.Wf0=0 #Initial Fruit Weight
        self.WFm0=0   #Initial Mature Fruit Weight


        for k, v in kwargs.items():
            setattr(self, k, v)

    def copy_with(self, **changes):
        p = TomgroParams(**vars(self))
        for k, v in changes.items():
            setattr(p, k, v)
        return p

In [4]:
bounds = {
    "N_m": (0.2, 1.0),
    "delta": (0.01, 0.08),
    "Beta": (0.05, 0.4),
    "N_b": (5, 30),
    "alpha_f": (0.3, 0.95),
    "vartheta": (0.05, 0.4),
    "T_Crit": (20, 28),
    "V_max": (2, 20),
    "N_FF": (8, 30),
    "kappa_f": (2, 10),
    "D_Fmax": (0.01, 0.15),
    "T_base_N":(6,11),
    "T_mid_N":(11.5,16),
    "T_opt_N":(22,30),
    "Tmax_N":(32,40),
    "T_base_Pg":(5,11),
    "T_mid_Pg":(11.5,16),
    "T_opt_Pg":(22,30),
    "Tmax_Pg":(32,42),
    "T_base_fruit":(6,11),
    "Tmid_fruit":(11.5,18),
    "T_opt_fruit":(20,28),
    "T_max_fruit":(30,40),
    "LAI_max":(3,5),
    "E":(0.63,1.2),
    "r_m":(0.0066,0.017),
    "Q_e":(0.055,0.09),
    "K":(0.4,0.9)
}

#Section 2 Calculating Rates and Supporting Functions

taken from Jones et al 1999 and Jones et al 1991

Tried to make it as modular as possible. This may present some challenges later on due to different timescales. Some functions for example are hourly, whereas others are daily. In main I will likely need both a hourly and daily loop. Data will also likely have to be structured a certain way.

## The Vegatative Block

 LAI is tightly coupled to Nodes. Thus, it is programmed as such. In one pass, both N and LAI will be calculated together. The difficulty is is that N uses hourly temperature measurements, whereas LAI uses daily. I will see how this works.


LAI = ρ · (δ/β) · ln{1 + exp[β·(N − N_b)]}


Eq 4 (the actual state update):
d(LAI)/dt =

 ρ · δ · λ(T_d) · exp[β(N−N_b)] / (1+exp[β(N−N_b)]) · dN/dt      

 if LAI ≤ LAI_max


d(LAI)/dt = 0                                                                if LAI > LAI_max

First Defining Temperature limitation (lamda(T_d) (average daily temp))


## Calculating Temperature inhibition functions for LAI and Nodes


First, both have their own seperate functions for temperature inhibition f_n(T) for Nodes. and Theta(T_d) for LAI. They differ in timescale, and theta is never defined, so for now I made it up lol.



In [5]:

def f_N(T,p):
  #Keeping the function dynamic, calculate the slope with changing Temperature optimums
  slope= 0.55/(p.T_mid_N-p.T_base_N)
  slope1=(1-0.55)/(p.T_opt_N-p.T_mid_N)
  slope2=-1/(p.Tmax_N-p.T_opt_N)

  if T<=p.T_base_N:
    return 0
  #Value= y1+(slope)*(T-T1(base or opt))
  elif T<=p.T_mid_N:
    return 0+slope*(T-p.T_base_N)
  elif T<=p.T_opt_N:
    return 0.55+slope1*(T-p.T_mid_N)
  else:
    return min(1,max(0,1+slope2*(T-p.T_opt_N)))

f_N(35,TomgroParams())






0

In [6]:
def lambda_Td(T_d, p):
    """Heat suppression of leaf expansion. 1 below T_Crit_LAI, linear to 0 at Tmax_LAI."""
    return float(np.clip((p.Tmax_LAI - T_d) / (p.Tmax_LAI - p.T_Crit_LAI), 0.0, 1.0))

In [7]:
def calc_vegetative(p, daily_data, current_N, current_LAI):
    temps_today = daily_data['Temperature'].values
    T_mean = np.mean(temps_today)

    # Node Development (dN/dt)
    f_vals = [f_N(T, p) for T in temps_today]
    dNdt = p.N_m * np.mean(f_vals)

    # LAI Growth (dLAI/dt)
    if current_LAI <= p.LAI_max:
        lambda_val = lambda_Td(T_mean, p)
        exponent = p.Beta * (current_N - p.N_b)
        logistic_term = np.exp(exponent) / (1 + np.exp(exponent))
        dLAIdt = p.rho * p.delta * lambda_val * logistic_term * dNdt
    else:
        dLAIdt = 0.0

    return dNdt, dLAIdt

# The Carbon Allocation Block
Calculating Total Weight, Mature Fruit Weight, and Fruit Weight

Starting off with PGRED (Photosynthesis Temperature Inhibition Function)  

P_g (Photosynthesis)
Then R_M (Respiration)
Then GR_net (Above Ground Growth)


## Calculating above ground biomass

Also known as GRnet


## Inhibition Functions
  PGRED Photosynthesis temperature inhibition function
  Root Partitioning fraction (function of nodes)

In [8]:
def Pgred(Hourly_T,p):
  slope=(0.67)/p.T_base_Pg
  slope1=(1-0.67)/(p.T_mid_Pg-p.T_base_Pg)
  slope2=0/(p.T_opt_Pg-p.T_mid_Pg)
  slope3=-1/(p.Tmax_Pg-p.T_opt_Pg)
#Value= y1+(slope)*(T-T1(base or opt))
  if Hourly_T<=p.T_base_Pg:
    return min(1,max(0,0+slope*(Hourly_T-0)))
  elif Hourly_T<=p.T_mid_Pg:
    return min(1,max(0,0.67+slope1*(Hourly_T-p.T_base_Pg)))
  elif Hourly_T<=p.T_opt_Pg:
    return min(1,max(0,1+slope2*(Hourly_T-p.T_mid_Pg)))
  else:
    return min(1,max(0,1+slope3*(Hourly_T-p.T_opt_Pg)))

Pgred(20,TomgroParams())

1

In [9]:
#Rooting partition fraction
def fr(Nodes):
  slope=(0.15-0.2)/(12-1)
  slope1=(0.1-0.15)/(21-12)
  slope2=(0.07-0.1)/(30-21)
  #Value= y1+(slope)*(X-X1)
  if Nodes<=1:
    return 0.2
  elif Nodes<=12:
    return 0.2+slope*(Nodes-1)
  elif Nodes<=21:
    return 0.15+slope1*(Nodes-12)
  elif Nodes<=30:
    return 0.1+slope2*(Nodes-21)
  else:
    return 0.07

print(fr(35))


0.07


## Calculating Assimilate Production and consumption
Pg (photosynthesis)
Rm (Respiration Rate)

In [10]:
#Calculating Photosynthesis
def calc_Pg(p, daily_data, current_LAI):
    D = p.D/24
    Q_e = p.Q_e
    K = p.K

    # Initiate Daily pg
    pg_daily = 0.0

    for _, hour in daily_data.iterrows():
        # Get the current climate conditions of this hour
        co2 = hour["CO2"]
        temperature = hour["Temperature"]
        ppfd = hour["PPFD"]

        if ppfd <= 0:
            pg_hourly = 0.0
        else:
            LF_max = co2 * p.tau
            numerator_1 = D * LF_max * Pgred(temperature, p)
            denominator_1 = K
            term1 = numerator_1 / denominator_1

            numerator_2 = (1 - p.m) * LF_max + Q_e * K * ppfd
            denominator_2 = (1 - p.m) * LF_max + Q_e * K * ppfd * np.exp(-K * current_LAI)
            term2 = np.log(numerator_2 / denominator_2)

            pg_hourly = term1 * term2

        # pg is summed across the day
        pg_daily += pg_hourly

    return pg_daily

In [11]:
#Calculating Respiration
def calc_Rm(p, daily_data, W, W_m):
    # Initiate Daily rm
    rm_daily = 0

    # Convert daily r_m to an hourly rate
    hourly_rm = p.r_m / 24.0

    for _, hour in daily_data.iterrows():
        temperature = hour["Temperature"]

        # Use Q10 for respiration sensitivity
        temp_sens = p.Q10 ** ((temperature - 20) / 10)

        # Calculate hourly Rm and prevent negative mass balance
        rm_hourly = temp_sens * hourly_rm * max(W - W_m, 0)

        rm_daily += rm_hourly

    return rm_daily

In [12]:
# Calculating above ground biomass (GRnet) produced
def calc_Grnet(p,pg,rm,N):
  return p.E*(pg-rm)*(1-fr(N))


## Calculating Fruit and crop Growth rate


Both W_F and W. g is the temperature inhibition function for daily temperature (fruit abortion and pollination) whereas f_f is for average daily temperatures

### Inhibition Functions

g, daily temperay inhibition (Occurs when temperatures in the day exclusively are too high ) I guess.

F_f: avg 24h temperature inhibition function




In [13]:
#Temperature inhibition for fruit abortion

def g(daily_climate,p):
  T_daytime=daily_climate.loc[daily_climate["PPFD"]>0,"Temperature"]
  mean_t=np.mean(T_daytime)
  if mean_t>=p.T_Crit:
    return min(1,max(0,1-0.154*(mean_t-p.T_Crit)))
  else:
    return 1


In [14]:
def f_f(T_d, p):
    if T_d <= p.T_base_fruit:
        return 0.0
    elif T_d <= p.T_opt_fruit:
        return (T_d - p.T_base_fruit) / (p.T_opt_fruit - p.T_base_fruit)
    elif T_d <= p.T_max_fruit:
        return (p.T_max_fruit - T_d) / (p.T_max_fruit - p.T_opt_fruit)
    else:
        return 0.0

### Calculate Dry Matter Production
Dwdt= Rate of W

DWfdt= Rate of Wf


In [15]:
def calc_dwdt(GR_net,p,dNdt,dW_fdt,LAI):
  if LAI>p.LAI_max:
    p1=p.p1
  else:
    p1=0
  dwdt=GR_net-p1*p.rho*dNdt #Source limited Growth
  dwdtmax=dW_fdt+(p.V_max-p1)*p.rho*dNdt #Sink limited Growth
  dwdt=min(dwdt,dwdtmax)
  return dwdt

In [16]:
#Calculating fruit growth rate, needs to be above minimum Nodes when fruit is 10 mm (weirdly specific)
def calc_dWf_dt(GR_net,p,g,N,f_f):
  if N> p.N_FF:
    dW_fdt= GR_net*p.alpha_f*(1-np.exp(-p.vartheta*(N-p.N_FF)))*f_f*g
  else :
    dW_fdt=0
  return dW_fdt


## Calculating Mature Fruit Weight

Function dF for fruit development, dwfm_dt calculates the "growth rate"of matuyre fruit.

In [17]:
def calc_d_f(p,Temperature_daily):
  d_f= p.D_Fmax* min(1,max(0,Temperature_daily-9)/19)
  return d_f



In [18]:
def calc_dwfm_dt(d_f,W_F,W_M,p,N):
  if N > p.N_FF+p.kappa_f:
    return d_f*(W_F-W_M)
  else:
    return 0

# Key Notes about the model

The functions for FF and Lambda were invented as there was no available literature about them. A Piecewise triangle was used for FF and a trapezoid was used for lambda.

Certain things like g, the daily temperature inhibition function, were adapted to the best of my abilities and intepretation of the paper.


#Section 3 Data Cleaning and Preperation





In [19]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Datasets/TomGRO Tomato Datasets") / "E1_1819"

CALIBRATION_FOLDERS = {
    "TreatA": BASE / "E1_1819_CultA_TreatA",
    "TreatB": BASE / "E1_1819_CultA_TreatB",
    "TreatC": BASE / "E1_1819_CultA_TreatC",
}
TEST_FOLDERS = {
    "TreatD": BASE / "E1_1819_CultA_TreatD",
}


ValueError: mount failed

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def clean_climate_data(folder):
  # Light Conversions
  Transmissivity= 0.5 # Transmissivity of the glass
  Lamp_PAR=215 #In Chamber E1
  PAR_Fraction = 0.47 # Fraction of PAR from the sun
  Joule_Photon_Conversion=4.6 # umol/J

  #Open the climate & Radiation files
  climate_file= next(folder.glob("*_Climate.xlsx")) #AB share a single file CD share another
  radiationfile= folder/ "E1_1819_Radiation.xlsx"

  climate= pd.read_excel(climate_file,header=0)
  radiation= pd.read_excel(radiationfile,header=0)

  #Change Temperature Name
  climate["Temperature"]= climate["Temp"]
  #Calculate PPFD From Lamp %
  climate["PPFD"]=(climate["Lamps"]/100)*Lamp_PAR
  #Merging Climate and Radiation Data
  merged= climate.merge(radiation,on="DateTime",how="inner")
  #Combining the radiation column with lamp column
  merged["PPFD"]= merged["PPFD"]+merged["Radiation"]*Transmissivity*PAR_Fraction*Joule_Photon_Conversion
  merged=merged.set_index("DateTime")[["Temperature","CO2","PPFD"]]
  merged=merged.dropna()
  return merged


TreatA_climate=clean_climate_data(Path(CALIBRATION_FOLDERS["TreatA"]))
TreatB_climate=clean_climate_data(Path(CALIBRATION_FOLDERS["TreatB"]))
TreatC_climate=clean_climate_data(Path(CALIBRATION_FOLDERS["TreatC"]))
TreatD_climate=clean_climate_data(Path(TEST_FOLDERS["TreatD"]))


In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 5),sharex=True)

TreatA_dailyavg= TreatA_climate.groupby(TreatA_climate.index.date).mean()
TreatB_dailyavg= TreatB_climate.groupby(TreatB_climate.index.date).mean()
TreatC_dailyavg= TreatC_climate.groupby(TreatC_climate.index.date).mean()
TreatD_dailyavg= TreatD_climate.groupby(TreatD_climate.index.date).mean()

axes[0,0].plot(TreatA_dailyavg.index,TreatA_dailyavg["Temperature"])
axes[0,0].set_title("Temperature A ")
axes[0,1].plot(TreatA_dailyavg.index,TreatA_dailyavg["CO2"])
axes[0,1].set_title("CO2 A")
axes[0,2].plot(TreatA_dailyavg.index,TreatA_dailyavg["PPFD"])
axes[0,2].set_title("PPFD A")

axes[1,0].plot(TreatB_dailyavg.index,TreatB_dailyavg["Temperature"])
axes[1,0].set_title("Temperature B ")
axes[1,1].plot(TreatB_dailyavg.index,TreatB_dailyavg["CO2"])
axes[1,1].set_title("CO2 B")
axes[1,2].plot(TreatB_dailyavg.index,TreatB_dailyavg["PPFD"])
axes[1,2].set_title("PPFD B")

axes[2,0].plot(TreatC_dailyavg.index,TreatC_dailyavg["Temperature"])
axes[2,0].set_title("Temperature C ")
axes[2,1].plot(TreatC_dailyavg.index,TreatC_dailyavg["CO2"])
axes[2,1].set_title("CO2 C")
axes[2,2].plot(TreatC_dailyavg.index,TreatC_dailyavg["PPFD"])
axes[2,2].set_title("PPFD C")

axes[3,0].plot(TreatD_dailyavg.index,TreatD_dailyavg["Temperature"])
axes[3,0].set_title("Temperature D ")
axes[3,1].plot(TreatD_dailyavg.index,TreatD_dailyavg["CO2"])
axes[3,1].set_title("CO2 D")
axes[3,2].plot(TreatD_dailyavg.index,TreatD_dailyavg["PPFD"])
axes[3,2].set_title("PPFD D")

plt.show()

In [ ]:
def fruit_weight_data(folder):
  '''
  Build a Sheet for Calibration of Wfm using measurements from the harvest sheet
  '''
  harvest_file= next(folder.glob("*_Harvest.xlsx"))
  harvest_df=pd.read_excel(harvest_file,header=0,sheet_name="Fruits")
  harvest_df=harvest_df[["Date","FruitDW"]].groupby("Date").mean()
  harvest_df["Cum_FruitDW"]=harvest_df["FruitDW"].cumsum()

  return harvest_df

TreatA_Harvest=fruit_weight_data(Path(CALIBRATION_FOLDERS["TreatA"]))
TreatB_Harvest=fruit_weight_data(Path(CALIBRATION_FOLDERS["TreatB"]))
TreatC_Harvest=fruit_weight_data(Path(CALIBRATION_FOLDERS["TreatC"]))
TreatD_Harvest=fruit_weight_data(Path(TEST_FOLDERS["TreatD"]))

TreatA_Harvest.head()

In [ ]:
def node_data(folder):
  node_file= next(folder.glob("*_Development.xlsx"))
  development_df=pd.read_excel(node_file,header=0,sheet_name="Leaf")
  development_df["Nodes"]=development_df["Leaves"]
  development_df=development_df[["Date","Nodes"]].groupby("Date").mean()
  return development_df

TreatA_Nodes=node_data(Path(CALIBRATION_FOLDERS["TreatA"]))
TreatB_Nodes=node_data(Path(CALIBRATION_FOLDERS["TreatB"]))
TreatC_Nodes=node_data(Path(CALIBRATION_FOLDERS["TreatC"]))
TreatD_Nodes=node_data(Path(TEST_FOLDERS["TreatD"]))

TreatA_Nodes.head()
  # In the dataset, two compartments are present, beta and gamma. Best to take an average of the two.


In [ ]:
def destructive_measurements(folder):
  plant_density=2.759
  destructive_file= next(folder.glob("*_Destructive.xlsx"))
  destructive_df=pd.read_excel(destructive_file,header=0)
  destructive_df["LAI_y"]=destructive_df["LeafArea"]*plant_density*(1/10000)
  destructive_df["W_actual"]=(destructive_df["LeafDW"]+destructive_df["FruitDW"]+destructive_df["StemDW"])*plant_density
  destructive_df=destructive_df[["Date","LAI_y","W_actual"]].groupby("Date").mean()
  return destructive_df

TreatA_Destructive=destructive_measurements(Path(CALIBRATION_FOLDERS["TreatA"]))
TreatB_Destructive=destructive_measurements(Path(CALIBRATION_FOLDERS["TreatB"]))
TreatC_Destructive=destructive_measurements(Path(CALIBRATION_FOLDERS["TreatC"]))
TreatD_Destructive=destructive_measurements(Path(TEST_FOLDERS["TreatD"]))

TreatA_Destructive.head()


#Section 4 the Main Function

In [ ]:
def runcropmodel(p,climate_df):
  N=p.N0
  W=p.W0
  Wf=p.Wf0
  WFm=p.WFm0
  LAI=p.LAI0
  record=[]
  for day, datetime in climate_df.groupby(climate_df.index.date):
    daily_data=climate_df.loc[datetime.index]

    #Calculate Vegetative States
    dNdt,dLAIdt=calc_vegetative(p,daily_data,N,LAI)
    #Biomass Production
    pg=calc_Pg(p,daily_data,LAI)
    rm=calc_Rm(p,daily_data,W,WFm)
    GR_net=calc_Grnet(p,pg,rm,N)

    #Biomass Allocation

    #Temperature inhibition functions
    g_val=g(daily_data,p)
    f_f_val=f_f(np.mean(daily_data["Temperature"]),p)
    #Calculate Rates
    #Fruit Growth Rate
    dwf_dt=calc_dWf_dt(GR_net,p,g_val,N,f_f_val)
    #Vegetative Growth Rate
    dw_dt=calc_dwdt(GR_net,p,dNdt,dwf_dt,LAI)

    #Fruit Maturity Rate
    d_f=calc_d_f(p,np.mean(daily_data["Temperature"]))
    dwfm_dt=calc_dwfm_dt(d_f,Wf,WFm,p,N)

    #Update States
    N+=dNdt
    LAI+=dLAIdt
    W+=dw_dt
    Wf+=dwf_dt
    WFm+=dwfm_dt
    record.append({"Date": day ,"N":N,"LAI":LAI,"W":W,"Wf":Wf,"WFm":WFm})

  return pd.DataFrame(record)


uncalibrated_Params=TomgroParams()



In [ ]:
def compare_results(crop_model,node_data,harvest_data,destructive_data):
  #Compare simulated model results to actual results.
  fig,axes= plt.subplots(nrows=4,ncols=1)
  axes[0].plot(crop_model["Date"],crop_model["N"],label="Simulated")
  axes[0].plot(node_data.index,node_data["Nodes"],label="Actual")
  axes[0].set_title("Nodes")
  axes[0].legend()

  axes[1].plot(crop_model["Date"],crop_model["LAI"],label="Simulated")
  axes[1].scatter(destructive_data.index,destructive_data["LAI_y"],label="Actual")
  axes[1].set_title("LAI")
  axes[1].legend()

  axes[2].plot(crop_model["Date"],crop_model["W"],label="Simulated")
  axes[2].scatter(destructive_data.index,destructive_data["W_actual"],label="Actual")
  axes[2].set_title("Weight")
  axes[2].legend()

  axes[3].plot(crop_model["Date"],crop_model["WFm"],label="Simulated")
  axes[3].plot(harvest_data.index,harvest_data["Cum_FruitDW"],label="Actual")
  axes[3].set_title("Fruit Weight")
  axes[3].legend()

model=runcropmodel(uncalibrated_Params,TreatA_climate)
compare_results(model,TreatA_Nodes,TreatA_Harvest,TreatA_Destructive)

# Section 5 Model Calibration

Objective Function: Weighted RMSE.

Chosen Parameters for calibration and prior range estimates derived from Sun et al 2025.

Method of calibration: Bayesian/ sklearn minimise.

Weigthed RMSE of W,Wfm, LAI & Nodes.

Wfm & Nodes given the most priority for two reasons.

Yield is the primary objective of the study. N also provides a good basis in the vegetative structure of the crop.

WFm & Nodes also have the most points, 17 each. Whereas LAI and W were destructive measurements and only have 2 points each across the season.


In [ ]:
#Start off with the objective function.
#weighted RMSE
def objective_function(x, all_params, tuned_params, datasets):
  #x is necessary because of the optimizer. It is the current list of parameters that
  #go in the optimization loop. all_params is not directly mutated.
  #datasets is a list of dictionairies of datasets

  param_updates = {tuned_params[i]: x[i] for i in range(len(tuned_params))}
  current_params = all_params.copy_with(**param_updates)

  #Weights for weighted RMSE
  N_loss_weight=3
  LAI_loss_weight=2
  Wfm_loss_weight=5
  W_loss_weight=2

  total_loss=0
  datasets_run=0

  for dataset in datasets:
    #Loop through a single dataset at a time. Take the total loss, and then divide by
    #Number of loops.
    #Define each dataset
    actual_nodes=dataset["Nodes"].reset_index()
    actual_LAI=dataset["LAI"].reset_index()
    actual_W=dataset["W"].reset_index()
    actual_WFm=dataset["WFm"].reset_index()

    #Loss for indivual dataset
    dataset_loss=0

    #Running the Model
    simulated_model=runcropmodel(current_params, dataset["climate"]).reset_index()

    # Ensure Date columns are datetime objects so merge works without throwing type errors
    simulated_model["Date"] = pd.to_datetime(simulated_model["Date"])
    actual_nodes["Date"] = pd.to_datetime(actual_nodes["Date"])
    actual_LAI["Date"] = pd.to_datetime(actual_LAI["Date"])
    actual_W["Date"] = pd.to_datetime(actual_W["Date"])
    actual_WFm["Date"] = pd.to_datetime(actual_WFm["Date"])

    #Merging the dataframes ensures that everything is in line
    merge_n=simulated_model.merge(actual_nodes,on="Date",how="inner")

    N_rmse=np.sqrt(np.mean((merge_n["N"]-merge_n["Nodes"])**2))
    dataset_loss+= N_rmse*N_loss_weight

    merge_LAI=simulated_model.merge(actual_LAI,on="Date",how="inner")
    LAI_rmse=np.sqrt(np.mean((merge_LAI["LAI"]-merge_LAI["LAI_y"])**2))
    dataset_loss+= LAI_rmse*LAI_loss_weight

    merge_W=simulated_model.merge(actual_W,on="Date",how="inner")
    W_rmse=np.sqrt(np.mean((merge_W["W"]-merge_W["W_actual"])**2))
    dataset_loss+= W_rmse*W_loss_weight


    merge_WFm=simulated_model.merge(actual_WFm,on="Date",how="inner")
    WFm_rmse=np.sqrt(np.mean((merge_WFm["WFm"]-merge_WFm["Cum_FruitDW"])**2))
    dataset_loss+= WFm_rmse*Wfm_loss_weight

    total_loss+=dataset_loss
    datasets_run+=1


  return total_loss/datasets_run if datasets_run > 0 else 1e65



---



In [ ]:
#Bayesian Optimization. Minimize.
from scipy.optimize import minimize

#List of Dictionairies of Dataframes.

calibration_datasets= [{"climate":TreatA_climate,"Nodes":TreatA_Nodes,"LAI":TreatA_Destructive,"W":TreatA_Destructive,"WFm":TreatA_Harvest},
                       {"climate":TreatB_climate,"Nodes":TreatB_Nodes,"LAI":TreatB_Destructive,"W":TreatB_Destructive,"WFm":TreatB_Harvest},
                       {"climate":TreatC_climate,"Nodes":TreatC_Nodes,"LAI":TreatC_Destructive,"W":TreatC_Destructive,"WFm":TreatC_Harvest}]

base_parameters=TomgroParams()
target_params=["E","K","rm","Qe"]
initial_guess=[base_parameters.E,base_parameters.K,base_parameters.r_m,base_parameters.Q_e]
bounds_list=[bounds["E"],bounds["K"],bounds["r_m"],bounds["Q_e"]]
result=minimize(objective_function,initial_guess,args=(base_parameters,target_params,calibration_datasets),bounds=bounds_list)

print(result.x)


In [ ]:
optimized_params=TomgroParams(E=result.x[0],K=result.x[1],r_m=result.x[2],Q_e=result.x[3])
model=runcropmodel(optimized_params,TreatD_climate)
compare_results(model,TreatA_Nodes,TreatD_Harvest,TreatD_Destructive)